In [ ]:
import runpy
import shutil
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader

import fluidbench
from fluidbench.cfdataset import CFDataset

In [ ]:
ID = "cylinder_Re41_raw"
dataset_path = fluidbench.download(ID)

In [ ]:
# Unstructured benchmark: each snapshot holds one value per finite-volume cell, fields are (variables, cells).
# The mesh (cell centres, cell -> vertex indices, vertices) is in constants.h5 for models that need it.
with h5py.File(dataset_path / "train" / "constants.h5", "r") as file:
    centers, indices, vertices = file["centers"][:], file["indices"][:], file["vertices"][:]
with h5py.File(dataset_path / "train" / "0.h5", "r") as file:
    scalars, fields = file["scalars"][:], file["fields"][:]
print(f"{len(centers)} cells, {len(vertices)} vertices; scalars {scalars.shape}, fields {fields.shape}")

In [ ]:
# A single training trajectory: validate on its tail (pairs snapshot i -> i + 1).
VAL_FRACTION = 0.1
n_pairs = len(fields) - 1
n_validation = max(1, round(n_pairs * VAL_FRACTION))
pairs = lambda indices: CFDataset(scalars[:-1], fields[:-1], fields[1:], indices)
train_dataset = pairs(np.arange(n_pairs - n_validation))
validation_dataset = pairs(np.arange(n_pairs - n_validation, n_pairs))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)
print(f"{len(train_dataset)} train pairs, {len(validation_dataset)} validation pairs")

In [ ]:
def train_and_bench(build, submission_id, name, description):
    torch.manual_seed(0)
    model = build()
    model.to(model.device)
    history = model.fit(train_loader, validation_loader)

    weights_path = Path("weights") / ID / f"{type(model).__name__}.pt"
    weights_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_weights(weights_path)

    submission_config = {
        "datasetId": ID,
        "id": submission_id,
        "model": name,
        "description": description,
        "authors": "Pablo A.C.",
        "year": 2026,
        "representation": "Graph",
    }
    training_history = {
        "train_error_history": history["train_losses"],
        "val_error_history": history["val_losses"],
        "train_error_name": history["loss_name"],
        "val_split": VAL_FRACTION,
    }
    return fluidbench.run(model, submission_config, training_history)

In [ ]:
from models.minimal_models import BaselineSame
submission = train_and_bench(BaselineSame, "baseline-same", "Baseline Same",
                             "Persistence baseline: every predicted snapshot is a copy of the input snapshot.")
submission["metrics"]["rolloutRmse"]

In [ ]:
from models.minimal_models import BaselineZero
submission = train_and_bench(BaselineZero, "baseline-zero", "Baseline Zero",
                             "Zero baseline: every predicted snapshot is an all-zero field.")
submission["metrics"]["rolloutRmse"]

In [ ]:
# Feed-forward network on the whole flattened snapshot: 3 hidden layers (512, 256, 512), ~25M parameters,
# almost all in the first and last layers (3 x 8000 values in and out).
from models.mlp import FieldMLP
submission = train_and_bench(lambda: FieldMLP(fields.shape[1:]), "mlp", "MLP",
                             "Feed-forward network (3 hidden layers) on the flattened snapshot, predicting the normalised increment.")
submission["metrics"]["rolloutRmse"]

In [ ]:
# Publish: copy this benchmark's submissions into the site's submissions folder and refresh its manifest.
published = Path("..") / "submissions" / ID
shutil.copytree(Path("submission") / ID, published, dirs_exist_ok=True)
runpy.run_path(str(Path("..") / "submissions" / "update.py"), run_name="__main__")
sorted(path.parent.name for path in published.glob("*/submission.json"))